# Australia Property Pipeline - NSW Land Registry -> DB columns
Raw NSW Valuer General sales file (~1M+ rows) -> cleaned CSV whose columns match `DB_SEQUENCE` exactly (same names, same order).

**Run top to bottom.** Only the CONFIG cell needs editing.

Fixes versus the previous notebook:
- Raw dates are `DD-MM-YYYY`; the old code parsed `%Y-%m-%d` (silently wrong / NaT). Now multi-format, day-first.
- Old Step 8 re-read the CSV into `df_clean` (no derived columns) so the final output had **0 columns**. Now one DataFrame flows through the whole notebook.
- `is_duplicate` is now actually created (it was referenced but never built).
- Dedup no longer collapses rows where `Dealing number` is blank.
- Output columns come from `DB_SEQUENCE`; columns with no NSW source are written empty.
- IDs / post codes are read as text (no more `2296.0`).
- Excel export removed: Excel's limit is 1,048,576 rows and this data is larger.
- Area: `Area (m2)`, `Area (source)`, `Area type (source)` are carried through as-is (appended after the 62 DB columns). `net_carpet_area_sqmt` = `Area (source)` x 10,000 when type is hectare (H), as-is when square metres (M); `Area (m2)` is only a fallback.

In [30]:
import gc, os, sys
import numpy as np
import pandas as pd

# ------------------------------- CONFIG -------------------------------
INPUT_PATH       = r"D:\Nilesh\Feasibility\Autralia_Sales_data\austrailia\Aus_property_data_raw.csv"
OUTPUT_PATH      = "australia_processed_01.csv"
DROP_DUPLICATES  = True    # True: remove repeats. False: keep them, flagged in is_duplicate
# ----------------------------------------------------------------------
print("Config OK")

Config OK


## Step 2: Load raw data

In [31]:
import csv, io, re

REQUIRED_RAW = [
    'District code','Property ID','Valuation number','Sale counter','Download date / time',
    'Property name','Property unit number','Property house number','Property street name',
    'Property locality','Property post code','Area (m2)','Area (source)','Area type (source)',
    'Dimensions','Contract date','Settlement date','Purchase price','Zoning','Nature of property',
    'Nature of property (description)','Primary purpose','Primary purpose (normalised)',
    'Strata lot number','Component code','Sale code','% Interest of sale','Dealing number',
    'Parcels in sale','Property legal description','Part interest flag','Non-market price flag',
    'Date error flag',
]

def _norm(name):
    """Header key: drop BOM / quotes / extra spaces, lower-case."""
    s = str(name).replace('﻿', '').strip().strip('"\'').strip()
    return re.sub(r'\s+', ' ', s).lower()

def sniff_file(path, nbytes=65536):
    """Detect encoding and delimiter from the start of the file."""
    with open(path, 'rb') as f:
        head = f.read(nbytes)
    if head[:2] in (b'\xff\xfe', b'\xfe\xff'):
        enc = 'utf-16'
    elif head[:3] == b'\xef\xbb\xbf':
        enc = 'utf-8-sig'
    else:
        try:
            head[:-4].decode('utf-8'); enc = 'utf-8'
        except UnicodeDecodeError:
            enc = 'cp1252'
    first_line = head.decode(enc, errors='replace').lstrip('﻿').splitlines()[0]
    counts = {d: first_line.count(d) for d in [',', '\t', ';', '|']}
    sep = max(counts, key=counts.get)
    return enc, sep, head[:200]

enc, sep, head_bytes = sniff_file(INPUT_PATH)
print(f'Detected encoding={enc!r}  delimiter={sep!r}')

READ_KW = dict(dtype=str, low_memory=False, on_bad_lines='skip', sep=sep, encoding=enc)
# Everything is read as text so IDs / post codes / dealing numbers are never mangled.
raw = pd.read_csv(INPUT_PATH, **READ_KW)

# Some tools wrap every whole row in quotes -> pandas sees ONE column. Unwrap and re-read.
if raw.shape[1] == 1 and sep in str(raw.columns[0]):
    print('Whole-line quoting detected - unwrapping rows and re-reading...')
    with open(INPUT_PATH, 'r', encoding=enc, newline='') as f:
        fixed = '\n'.join(
            ln[1:-1].replace('""', '"') if (ln.startswith('"') and ln.endswith('"')) else ln
            for ln in (l.rstrip('\r\n') for l in f)
        )
    raw = pd.read_csv(io.StringIO(fixed), **{k: v for k, v in READ_KW.items() if k != 'encoding'})
    del fixed

# Normalise header text (BOM, stray quotes/spaces, case) to the exact expected names
_req = {_norm(c): c for c in REQUIRED_RAW}
raw.columns = [_req.get(_norm(c), str(c).strip()) for c in raw.columns]

missing = [c for c in REQUIRED_RAW if c not in raw.columns]
if missing:
    # If the header is absent, locate the first schema-shaped data row and skip any preamble.
    skip_rows = None
    with open(INPUT_PATH, 'r', encoding=enc, newline='') as f:
        reader = csv.reader(f, delimiter=sep)
        previous_end_line = 0
        for row in reader:
            start_line = previous_end_line + 1
            previous_end_line = reader.line_num
            if len(row) == len(REQUIRED_RAW) and row and re.fullmatch(r'\d+', row[0].strip()):
                skip_rows = start_line - 1
                break

    if skip_rows is not None:
        print(f'Raw header not found; treating rows as {len(REQUIRED_RAW)} columns and skipping {skip_rows} preamble lines.')
        raw = pd.read_csv(
            INPUT_PATH,
            **READ_KW,
            header=None,
            names=REQUIRED_RAW,
            skiprows=skip_rows,
        )
        raw.columns = [_req.get(_norm(c), str(c).strip()) for c in raw.columns]

# Approximate check for rows dropped by on_bad_lines='skip'
with open(INPUT_PATH, 'rb') as f:
    file_lines = sum(1 for _ in f) - 1
print(f'Loaded rows : {len(raw):,}   (file lines ~{file_lines:,}; a large gap means bad rows were skipped)')
print(f'Columns     : {raw.shape[1]}')

missing = [c for c in REQUIRED_RAW if c not in raw.columns]
extra   = [c for c in raw.columns if c not in REQUIRED_RAW]
print('Missing raw columns:', missing or 'none')
print('Unexpected extra columns:', extra or 'none')
if missing:
    print('\nFirst bytes of file :', head_bytes)
    print('Columns pandas found:', list(raw.columns)[:40])
    print('INPUT_PATH          :', INPUT_PATH)
    raise KeyError(f'Raw file is missing columns: {missing}')
raw.head(3)


Detected encoding='utf-8-sig'  delimiter=','
Raw header not found; treating rows as 33 columns and skipping 9 preamble lines.
Loaded rows : 1,278,797   (file lines ~1,278,805; a large gap means bad rows were skipped)
Columns     : 33
Missing raw columns: none
Unexpected extra columns: none


,District code,Property ID,Valuation number,Sale counter,Download date / time,Property name,Property unit number,Property house number,Property street name,Property locality,...,Strata lot number,Component code,Sale code,% Interest of sale,Dealing number,Parcels in sale,Property legal description,Part interest flag,Non-market price flag,Date error flag
0,258,1436634.0,NaN,22,2020-01-13 01:02:00,NaN,NaN,28,Remly St,Roselands,...,NaN,XRR,NaN,NaN,AP812259,1,30/4720,False,False,False
1,259,57757.0,NaN,21,2020-01-13 01:02:00,NaN,NaN,14,Julian Rd,Kincumber,...,NaN,GHC,NaN,NaN,AP818279,1,23/701415,False,False,False
2,267,1970871.0,NaN,11,2020-01-13 01:03:00,NaN,NaN,34,Westbourne St,Stanmore,...,NaN,ETL,NaN,NaN,AP815826,1,19/1096305,False,False,False


## Step 3: Mapping dictionaries

In [32]:
# Nature Code and Primary Purpose Mappings as per DB1 specification
NATURE_MAP = {
    'R': 'Residential',
    'C': 'Commercial',
    'I': 'Industrial',
    'A': 'Agricultural',
    'M': 'Mixed Use',
    'V': 'Vacant Land',
}

print('Mapping dictionaries ready')


Mapping dictionaries ready


## Step 4: Cleaning helpers
Dates in the raw file are `DD-MM-YYYY`. The parser tries day-first formats first, then ISO.

In [33]:
def clean_text(s):
    if s is None:
        return pd.Series(dtype='string')
    s = s.astype('string').str.strip()
    return s.mask((s == '') | (s.str.lower() == 'nan'))

def clean_id(s):
    """Text id with any float artefact ('212135.0') removed."""
    return clean_text(s).str.replace(r'\.0+$', '', regex=True)

def parse_date(s):
    s = clean_text(s)
    out = pd.Series(pd.NaT, index=s.index, dtype='datetime64[ns]')
    for fmt in ('%Y-%m-%d', '%d-%m-%Y', '%d/%m/%Y', 'ISO8601'):
        todo = out.isna() & s.notna()
        if not todo.any():
            break
        out[todo] = pd.to_datetime(s[todo], format=fmt, errors='coerce')
    return out

def to_bool(s):
    if s is None:
        return pd.Series(False, index=raw.index, dtype='boolean')
    return clean_text(s).str.upper().map({'TRUE': True, 'FALSE': False, '1': True, '0': False}).astype('boolean')

def map_project_type(purpose_s):
    purpose = clean_text(purpose_s).fillna('').astype(str).str.strip().str.lower()
    conditions = [
        purpose.str.contains(r'residen', regex=True),
        purpose.str.contains(r'commerci', regex=True),
    ]
    choices = ['Residential', 'Commercial']
    return pd.Series(np.select(conditions, choices, default='Others'), index=purpose_s.index)

def map_property_type(purpose_s):
    purpose = clean_text(purpose_s).fillna('').astype(str).str.strip().str.lower()
    conditions = [
        purpose.str.contains(r'\bflats?\b', regex=True),
        purpose.str.contains(r'\bshops?\b|shopfront|shopping', regex=True),
        purpose.str.contains(r'\boffices?\b|officebuilding', regex=True),
        purpose.str.contains(r'\bvillas?\b', regex=True),
        purpose.str.contains(r'\bplots?\b', regex=True),
    ]
    choices = ['flat', 'shop', 'office', 'villa', 'plot']
    return pd.Series(np.select(conditions, choices, default='others'), index=purpose_s.index)

print('Helpers ready')


Helpers ready


## Step 5: Transform raw -> DB columns

In [34]:
out = pd.DataFrame(index=raw.index)

# 1. index: Property ID (cleaned of float .0 artifact)
out['index'] = clean_id(raw.get('Property ID'))

# 2. project_name: Property name
out['project_name'] = clean_text(raw.get('Property name'))

# 3. location_name: Property locality
locality = clean_text(raw.get('Property locality'))
out['location_name'] = locality

# 4. agreement_price: Purchase price(AUD)
price_col = 'Purchase price' if 'Purchase price' in raw.columns else 'Purchase price(AUD)'
out['agreement_price'] = pd.to_numeric(raw.get(price_col), errors='coerce')

# 5. transaction_date: Settlement date
txn_date = parse_date(raw.get('Settlement date'))
out['transaction_date'] = txn_date

# 6. net_carpet_area_sqmt  +  the 3 raw area columns carried through exactly as in the raw file
RAW_PRESERVE_COLS = [
    'Area (m2)', 'Area (source)', 'Area type (source)',
    'Nature of property', 'Nature of property (description)',
    'Primary purpose', 'Primary purpose (normalised)'
]
RAW_AREA_COLS = ['Area (m2)', 'Area (source)', 'Area type (source)']

area_m2_raw = pd.to_numeric(raw['Area (m2)'], errors='coerce')
area_src    = pd.to_numeric(raw['Area (source)'], errors='coerce')
area_type   = clean_text(raw['Area type (source)']).str.upper()

HECTARE_CODES = ['H', 'HA', 'HECTARE', 'HECTARES']
SQM_CODES     = ['M', 'M2', 'SQM']

is_ha  = area_type.isin(HECTARE_CODES).fillna(False).to_numpy(dtype=bool)
is_sqm = area_type.isin(SQM_CODES).fillna(False).to_numpy(dtype=bool)

# Hectares -> x 10,000 ; square metres -> as is ; unknown/blank type -> NaN (not guessed)
area_from_source = pd.Series(
    np.select([is_ha, is_sqm], [area_src.to_numpy() * 10000.0, area_src.to_numpy()], default=np.nan),
    index=raw.index,
)

# Source + type is the primary rule; 'Area (m2)' only fills where it can't be derived
final_area = area_from_source.fillna(area_m2_raw)
out['net_carpet_area_sqmt'] = np.round(final_area, 4)

# Raw columns, untouched (original text values)
for _c in RAW_PRESERVE_COLS:
    out[_c] = raw[_c]

# Diagnostic: does the raw 'Area (m2)' agree with source x hectare conversion?
both = area_from_source.notna() & area_m2_raw.notna()
bad  = both & ((area_from_source - area_m2_raw).abs() > 1)
print(f'Area type counts : H={is_ha.sum():,}  M={is_sqm.sum():,}  other/blank={(~(is_ha | is_sqm)).sum():,}')
print(f"Rows where 'Area (m2)' != converted source: {bad.sum():,} of {both.sum():,}")
if bad.any():
    print(raw.loc[bad, RAW_AREA_COLS].head(10))

# 7. project_type: if residential / commercial found in property_type_raw, else Others
out['property_type_raw'] = clean_text(raw.get('Primary purpose'))
out['project_type'] = map_project_type(out['property_type_raw'])

# 8 & 9. sro_code & sro_name
out['sro_code'] = np.nan
out['sro_name'] = np.nan

# 10. document_no: Dealing number
out['document_no'] = clean_text(raw.get('Dealing number'))

# 11. transaction_type
out['transaction_type'] = np.nan

# 12. city_name: Property locality
out['city_name'] = locality

# 13. market_value
out['market_value'] = np.nan

# 14. property_description: Property legal description
out['property_description'] = clean_text(raw.get('Property legal description'))

# 15. floor_number
out['floor_number'] = np.nan

# 16. unit_number: Property unit
unit_col = 'Property unit number' if 'Property unit number' in raw.columns else 'Property unit'
out['unit_number'] = clean_id(raw.get(unit_col))

# 17. property_type_raw: populated from raw 'Primary purpose' (assigned at step 7)

# 18. property_type: flat, shop, office, villa, plot found in property_type_raw, else others
out['property_type'] = map_property_type(out['property_type_raw'])
# 19. year: Settlement date year
out['year'] = txn_date.dt.year.astype('Int64')

# 20 & 21. balcony_sqmt & terrace_sqmt
out['balcony_sqmt'] = np.nan
out['terrace_sqmt'] = np.nan

# 22 & 23. seller_name & purchaser_name
out['seller_name'] = np.nan
out['purchaser_name'] = np.nan

# 24. transaction_category: Sales
out['transaction_category'] = 'Sales'

# 25 to 31. internaldocumentnumber, micrno, bank_type, party_code, dateofexecution, stampdutypaid, registrationfees
out['internaldocumentnumber'] = np.nan
out['micrno'] = np.nan
out['bank_type'] = np.nan
out['party_code'] = np.nan
out['dateofexecution'] = np.nan
out['stampdutypaid'] = np.nan
out['registrationfees'] = np.nan

# 32 to 35. project_lat, project_lng, location_lat, location_lng
# Coordinates must be numeric GPS Latitude/Longitude, NOT address strings.
# NSW Land Registry raw data does not contain GPS coordinates, so they remain null.
out['project_lat'] = np.nan
out['project_lng'] = np.nan
out['location_lat'] = np.nan
out['location_lng'] = np.nan

# 36. quarter: Settlement date quarter
q_str = ('Q' + txn_date.dt.quarter.astype('Int64').astype('string')
         + '-' + txn_date.dt.year.astype('Int64').astype('string'))
out['quarter'] = q_str.mask(txn_date.isna())

# 37 to 41. bhk_br, buyer_pincode, locality_of_buyer, buyer_district, buyer_statename
out['bhk_br'] = np.nan
out['buyer_pincode'] = np.nan
out['locality_of_buyer'] = np.nan
out['buyer_district'] = np.nan
out['buyer_statename'] = np.nan

# 42 & 43. llm_processed & manual_processed: no
out['llm_processed'] = 'no'
out['manual_processed'] = 'no'

# 44 to 46. tower, is_duplicate, primary_sale_or_secondary_sale
out['tower'] = np.nan
out['is_duplicate'] = np.nan
out['primary_sale_or_secondary_sale'] = np.nan

# 47 & 48. country & state
out['country'] = 'Austrailia'
out['state'] = 'North South Wales'

# 49. micromarket
out['micromarket'] = np.nan

# 50. sub_locatlity: Property street name
out['sub_locatlity'] = clean_text(raw.get('Property street name'))

# 51. pincode: Property post code
pin = clean_id(raw.get('Property post code'))
out['pincode'] = pin.where(pin.isna(), pin.str.zfill(4))

# 52 to 56. parking_count, facing_direction, view_type, furnishing_status, condition_status
out['parking_count'] = np.nan
out['facing_direction'] = np.nan
out['view_type'] = np.nan
out['furnishing_status'] = np.nan
out['condition_status'] = np.nan

# 57 & 58. source_accessibility & source_accessibility_way
out['source_accessibility'] = 'Easy'
out['source_accessibility_way'] = 'Download'

# 59 to 61. cost, time, data_type
out['cost'] = np.nan
out['time'] = np.nan
out['data_type'] = np.nan

# 62. data_source: nsw
out['data_source'] = 'nsw'

# Helper DataFrame for QA / duplicate checks
qa = pd.DataFrame({
    'property_id': out['index'],
    'non_market': to_bool(raw.get('Non-market price flag')),
    'part_interest': to_bool(raw.get('Part interest flag')),
    'date_error': to_bool(raw.get('Date error flag')),
}, index=raw.index)

print(f'Transformed raw records: {len(out):,}')
print(out['project_type'].value_counts(dropna=False))


Area type counts : H=81,104  M=931,884  other/blank=265,809
Rows where 'Area (m2)' != converted source: 0 of 1,012,901
Transformed raw records: 1,278,797
project_type
Residential    1098391
Others          148971
Commercial       31435
Name: count, dtype: int64


## Step 6: Duplicate detection
Key = `property_id + document_number + agreement_price + transaction_date`. First occurrence is kept.
Rows with a blank dealing number are **not** treated as duplicates of each other.

In [35]:
key_df = pd.DataFrame({
    'property_id': out['index'],
    'document_no': out['document_no'],
    'agreement_price': out['agreement_price'],
    'transaction_date': out['transaction_date'],
})
has_key = key_df['document_no'].notna() & key_df['property_id'].notna()
dup = key_df.duplicated(keep='first') & has_key

out['is_duplicate'] = dup.values
n_before, n_dup = len(out), int(dup.sum())
print(f'Rows before      : {n_before:,}')
print(f'Duplicates found : {n_dup:,} ({n_dup / n_before:.2%})')
print(f'Rows w/o dealing number (not deduped): {(~has_key).sum():,}')

if DROP_DUPLICATES:
    keep = ~dup.values
    out, qa = out.loc[keep], qa.loc[keep]
    out['is_duplicate'] = False
print(f'Rows after       : {len(out):,}')
del key_df, dup
gc.collect()


Rows before      : 1,278,797
Duplicates found : 19,526 (1.53%)
Rows w/o dealing number (not deduped): 150
Rows after       : 1,259,271


637

## Step 7: Final DataFrame (without DB sequence)
Keep all transformed & cleaned columns directly without forcing external DB columns.

In [36]:
# Create df_final directly from cleaned data without DB_SEQUENCE restriction
df_final = out.copy()

# Preserve 'Download date / time' from raw if present for downstream year/date derivation
if 'raw' in globals() and 'Download date / time' in raw.columns and 'Download date / time' not in df_final.columns:
    df_final['Download date / time'] = raw.loc[out.index, 'Download date / time']

print(f'df_final ready: {df_final.shape[1]} columns, {len(df_final):,} records')
df_final.head(3)

df_final ready: 70 columns, 1,259,271 records


,index,project_name,location_name,agreement_price,transaction_date,net_carpet_area_sqmt,Area (m2),Area (source),Area type (source),Nature of property,...,view_type,furnishing_status,condition_status,source_accessibility,source_accessibility_way,cost,time,data_type,data_source,Download date / time
0,1436634,<NA>,Roselands,487500,2020-01-06,538.8,538.8,538.8,M,R,...,NaN,NaN,NaN,Easy,Download,NaN,NaN,NaN,nsw,2020-01-13 01:02:00
1,57757,<NA>,Kincumber,342500,2020-01-09,892.2,892.2,892.2,M,R,...,NaN,NaN,NaN,Easy,Download,NaN,NaN,NaN,nsw,2020-01-13 01:02:00
2,1970871,<NA>,Stanmore,625000,2020-01-08,177.0,177.0,177.0,M,R,...,NaN,NaN,NaN,Easy,Download,NaN,NaN,NaN,nsw,2020-01-13 01:03:00


## Step 8: Data quality report

In [37]:
n = len(df_final)
print('=' * 64); print('DATA QUALITY REPORT'); print('=' * 64)
print(f'Records: {n:,}   Columns: {df_final.shape[1]}')

print('\n--- Fill rate (columns with data) ---')
fill = df_final.notna().mean().mul(100).round(2)
print(fill[fill > 0].sort_values(ascending=False).to_string())

print('\n--- Date range ---')
valid_dates = pd.to_datetime(df_final['transaction_date'], errors='coerce').dropna()
min_d = valid_dates.min().strftime('%Y-%m-%d') if not valid_dates.empty else 'N/A'
max_d = valid_dates.max().strftime('%Y-%m-%d') if not valid_dates.empty else 'N/A'
print(f'transaction_date : {min_d} -> {max_d}')
print(f"missing transaction_date: {df_final['transaction_date'].isna().sum():,}")

if 'quarter' in df_final.columns:
    print('\n--- Quarter distribution (chronological) ---')
    print(df_final['quarter'].dropna().value_counts().head(16).to_string())

if 'project_type' in df_final.columns:
    print('\n--- Project type ---')
    print(df_final['project_type'].value_counts(dropna=False).to_string())

if 'agreement_price' in df_final.columns:
    print('\n--- Price (AUD) ---')
    p = df_final['agreement_price']
    print(f'null: {p.isna().sum():,}   zero/negative: {(p <= 0).sum():,}   < $10k: {(p < 10_000).sum():,}')
    print(p.describe().apply(lambda v: f'{v:,.2f}').to_string())

if 'net_carpet_area_sqmt' in df_final.columns:
    print('\n--- Net carpet area (sq mt) ---')
    a = df_final['net_carpet_area_sqmt']
    print(f'null: {a.isna().sum():,}   zero: {(a == 0).sum():,}')
    print(a.describe().apply(lambda v: f'{v:,.2f}').to_string())
print('=' * 64)


DATA QUALITY REPORT
Records: 1,259,271   Columns: 70

--- Fill rate (columns with data) ---
agreement_price                     100.00
location_name                       100.00
data_source                         100.00
state                               100.00
transaction_date                    100.00
Nature of property                  100.00
project_type                        100.00
document_no                         100.00
city_name                           100.00
Nature of property (description)    100.00
property_description                100.00
is_duplicate                        100.00
llm_processed                       100.00
manual_processed                    100.00
quarter                             100.00
transaction_category                100.00
year                                100.00
property_type                       100.00
Download date / time                100.00
country                             100.00
source_accessibility_way            100.00
sourc

In [40]:
df_final.to_csv("processed.csv")